# Customer simulator: QLoRA training on Kaggle (T4)
Settings -> Accelerator: **GPU T4**, Internet: **on**. Add a Kaggle Secret named `HF_TOKEN` (write access).
Runs: prepare data -> train (only customer turns in the loss) -> save adapter + GGUF -> upload to Hugging Face.

In [ ]:
# 1. Install (GPU-only deps) and get the code
!pip install -q unsloth
!git clone https://github.com/TODO-USER/TODO-REPO.git code  # TODO: public repo URL
%cd code
!pip install -q --no-deps -e .

In [ ]:
# 2. Hugging Face token from Kaggle Secrets (never hard-code it)
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# 3. Data: download ABCD and build the customer-side SFT files
!mkdir -p data/raw
!wget -q -O data/raw/abcd_v1.1.json.gz https://github.com/asappresearch/abcd/raw/master/data/abcd_v1.1.json.gz  # TODO(verify) path
!gunzip -f data/raw/abcd_v1.1.json.gz
!python -m usersim.prepare_data --config configs/data.yaml
!cat data/processed/stats.json | head -20

In [ ]:
# 4. Token-length check before choosing max_seq_len (day 3)
from transformers import AutoTokenizer
import json, numpy as np
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
lens = [len(tok.apply_chat_template(json.loads(l)["messages"], tokenize=True)) for l in open("data/processed/train.jsonl")]
print("p50/p95/p99/max:", np.percentile(lens, [50, 95, 99]), max(lens))

In [ ]:
# 5a. Smoke test: 500 examples, 1 epoch (day 6)
!python -m usersim.train --config configs/train.yaml --max-examples 500 --epochs 1

In [ ]:
# 5b. Full training (day 8). Set hub_repo in configs/train.yaml to push the adapter.
# !python -m usersim.train --config configs/train.yaml --export-gguf

In [ ]:
# 6. Upload adapter + GGUF to Hugging Face (private until release)
from huggingface_hub import HfApi
api = HfApi()
repo = "TODO-USER/qwen3-4b-customer-sim"  # TODO
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path="outputs/train/qwen3-4b-usersim/adapter", repo_id=repo, path_in_repo="adapter")
# api.upload_folder(folder_path="outputs/train/qwen3-4b-usersim/gguf", repo_id=repo + "-GGUF")